# 02 - Live view: see the ACC work, and watch the experiments run
**Part A** is an interactive illustration (change the scenario, number of agents and fixed window, and the plot updates).
**Parts B-D** run the real RQ2 experiments in the background and redraw a live progress view every few seconds.

Everything is resumable: if Colab disconnects or you stop a cell, run it again and it continues where it stopped.
No GPU needed (Runtime -> Change runtime type -> CPU is fine).

In [ ]:
# 1) Setup: Drive, repo, dependencies   (run once per session)
import os, sys, subprocess, base64, tempfile
from pathlib import Path

REPO_URL = "https://github.com/notanut/eem-acc-skripsi.git"
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/eem-acc-skripsi"
    DRIVE_ROOT = "/content/drive/MyDrive/Skripsi"
    def git(*args, cwd=None):
        subprocess.run(["git", *args], cwd=cwd, check=True)
    if os.path.isdir(os.path.join(REPO_DIR, ".git")):
        git("pull", "--ff-only", cwd=REPO_DIR)
    else:
        git("clone", REPO_URL, REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements.txt", "-e", REPO_DIR], check=True)
else:                                   # running outside Colab (local test): use this checkout + a temp folder
    REPO_DIR = str(Path.cwd().parent) if Path.cwd().name == "notebooks" else str(Path.cwd())
    DRIVE_ROOT = tempfile.mkdtemp(prefix="eem_acc_")
    sys.path.insert(0, f"{REPO_DIR}/src")
sys.path.insert(0, f"{REPO_DIR}/experiments")
os.environ["EEM_ACC_DRIVE_ROOT"] = DRIVE_ROOT
from eem_acc.utils import Paths, set_seed
set_seed(42); Paths().ensure()
print("repo:", REPO_DIR, "\nresults folder:", DRIVE_ROOT)

## Part A - Interactive illustration (not a result)
ACC (using its *start values*, not tuned ones) against a fixed window, on one seed. Pick a scenario, the number of agents (M) and the fixed window.
Top: the window each method allows. Middle: how many attempts per second were throttled (429/503). Bottom: p95 server latency against the target (dashed).

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from eem_acc import scenarios as S
from eem_acc.viz import demo_figure

def show(scenario="CAP-drop", m_agents=16, fixed_w=16):
    fig = demo_figure(scenario, m_agents, seed=0, fixed_w=fixed_w)
    plt.show()

try:
    from ipywidgets import interact, Dropdown
    interact(show,
             scenario=Dropdown(options=list(S.SCENARIOS), value="CAP-drop", description="scenario"),
             m_agents=Dropdown(options=list(S.M_LEVELS), value=16, description="agents M"),
             fixed_w=Dropdown(options=[4, 8, 16, 32, 64], value=16, description="fixed W"));
except ImportError:
    show()

## Part B - Watch the dev phase run live
5,520 simulated runs (81 ACC settings + fixed windows + token bucket, 4 scenarios, 3 agent counts, 5 seeds). About 15 minutes on a fast machine; plan for roughly 15-40 minutes on Colab's CPU. The ETA is rough: the first runs are the light 4-agent ones, so it grows as heavier runs start.
Top: progress and ETA. Tiles: runs finished per scenario and agent count. Table: **partial** ranking of methods from the runs finished so far (it only becomes meaningful when the phase is complete).

Set `LIMIT` to a small number (e.g. 60) for a quick trial; keep `None` for the full run. If you stop the cell, run it again to continue.

In [ ]:
import json, threading, time
import matplotlib.pyplot as plt
from IPython.display import clear_output, display
import run_rq2
from eem_acc.analysis import load_results, score_methods
from eem_acc.monitor import snapshot
from eem_acc.viz import monitor_figure

ROOT = Path(DRIVE_ROOT)
LIMIT = None          # e.g. 60 for a quick trial
_WORKERS = {}         # one background worker per phase (re-running a cell never starts a duplicate)

def watch(phase, frozen=None, limit=None, refresh=5.0):
    w = _WORKERS.get(phase)
    if w is None or not w["thread"].is_alive():
        w = {"err": []}
        def work():
            try:
                run_rq2.run_phase(phase, ROOT, limit, frozen)
            except BaseException as e:
                w["err"].append(e)
        w["thread"] = threading.Thread(target=work, daemon=True)
        w["thread"].start()
        _WORKERS[phase] = w
    udir = ROOT / "results" / "rq2" / phase / "units"
    while True:
        alive = w["thread"].is_alive()
        snap = snapshot(ROOT, phase, frozen)
        clear_output(wait=True)
        fig = monitor_figure(snap); plt.show()
        df = load_results(udir) if udir.exists() and any(udir.glob("*.json")) else None
        if df is not None and len(df):
            cols = ["tag", "score", "mean_slo_violation", "mean_call_success", "n_cells"]
            print(f"Partial ranking from {len(df)} finished runs (final only when the phase is complete):")
            display(score_methods(df)[cols].head(8).round(3))
        if not alive:
            break
        time.sleep(refresh)
    if w["err"]:
        raise w["err"][0]
    print("finished.")

watch("dev", limit=LIMIT)

## Part C - Select the configuration (dev results only)
Picks the ACC settings, fixed window and token-bucket rate from the **dev** runs using the approved rule (maximise goodput subject to SLO violations <= 5% and call success >= 99%).
It refuses to run until the dev phase is complete, and it never relaxes the constraints on its own.

In [ ]:
code_ = run_rq2.do_select(ROOT)
print("exit code:", code_, "(0 = configuration chosen and saved; 2 = dev phase incomplete; 3 = no feasible candidate)")

## Part D - Watch the test phase run live
1,080 runs with the frozen configuration and fresh seeds (100-109). **Do not look at test results before the configuration is frozen.**

In [ ]:
frozen_path = (Path(REPO_DIR) / "configs" / "rq2_frozen.json")
if not frozen_path.exists():
    frozen_path = ROOT / "results" / "rq2" / "rq2_frozen.json"
if frozen_path.exists():
    frozen = json.loads(frozen_path.read_text())
    watch("test", frozen=frozen, limit=LIMIT)
else:
    print("No frozen configuration yet: finish Part B and run Part C first.")